In [1]:
import numpy as np


In [2]:
import sys
sys.path.append('../')
from pqcqec.noise.builder import build_regular_noisy_circuit, create_pqc_circuit_template, update_pqc_circuit_template
from pqcqec.circuits.generate import generate_random_circuit


In [3]:
NUM_QUBITS = 2
NUM_GATES = 4
NUM_GATE_BLOCKS = 2

# User controls PQC blocks exactly - no automatic final block
# PQC inserted after every NUM_GATE_BLOCKS gates
PQC_BLOCKS = NUM_GATES // NUM_GATE_BLOCKS  # 4 // 2 = 2

## Configuration

**PQC Placement Strategy:**
- `PQC_BLOCKS = NUM_GATES // NUM_GATE_BLOCKS` 
- PQC inserted **only** after every `NUM_GATE_BLOCKS` logical gates
- **No automatic final block** - you have full control!
  
For 4 gates with `gate_blocks=2`:
- PQC after gate 1 (index 1) ✓
- PQC after gate 3 (index 3) ✓
- Total: **2 PQC blocks**

In [4]:
circuit = generate_random_circuit(NUM_QUBITS, NUM_GATES, seed=42, backend='list')
print("Generated Circuit:")
print(circuit)

Generated Circuit:
[('cx', [0, 1], []), ('x', [1], []), ('z', [0], []), ('z', [0], [])]


In [5]:
x_noise = np.random.normal(0, 0.01, size=(NUM_GATES,))
z_noise = np.random.normal(0, 0.01, size=(NUM_GATES,))

noisy_circ = build_regular_noisy_circuit(circuit, x_noise=x_noise, z_noise=z_noise, return_tagged=True)
print("Noisy Circuit with Tagged Noise:")
print(noisy_circ)

Noisy Circuit with Tagged Noise:
[('cx', [0, 1], []), ('rx', [0], [np.float64(-0.00392327582779992)], {'noise': True}), ('rz', [0], [np.float64(0.017554795239014195)], {'noise': True}), ('rx', [1], [np.float64(-0.00392327582779992)], {'noise': True}), ('rz', [1], [np.float64(0.017554795239014195)], {'noise': True}), ('x', [1], []), ('rx', [1], [np.float64(-0.0004656090720974087)], {'noise': True}), ('rz', [1], [np.float64(-0.00936443227274573)], {'noise': True}), ('z', [0], []), ('rx', [0], [np.float64(0.01852641592424441)], {'noise': True}), ('rz', [0], [np.float64(0.01036728019993393)], {'noise': True}), ('z', [0], []), ('rx', [0], [np.float64(0.0014981455170642141)], {'noise': True}), ('rz', [0], [np.float64(0.0033150212600172543)], {'noise': True})]


In [9]:
pqc_noisy_circ_template = create_pqc_circuit_template(
    noisy_circ, num_qubits=NUM_QUBITS, gate_blocks=NUM_GATE_BLOCKS, 
    pqc_gates=['rz', 'rx', 'rz'], num_pqc_blocks=PQC_BLOCKS, dtype=np.float32, ignore_noise_gates=True)

print("PQC Circuit Template:")
print(pqc_noisy_circ_template)

PQC Circuit Template:
{'gate_ids': array([6, 3, 5, 3, 5, 0, 5, 3, 5, 5, 3, 5, 3, 5, 1, 3, 5, 1, 5, 3, 5, 5,
       3, 5, 3, 5], dtype=int32), 'wire1': array([0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 0, 0], dtype=int32), 'wire2': array([ 1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
       -1, -1, -1, -1, -1, -1, -1, -1, -1], dtype=int32), 'theta': array([ 0.        , -0.00392328,  0.0175548 , -0.00392328,  0.0175548 ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        , -0.00046561, -0.00936443,  0.        ,
        0.01852642,  0.01036728,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.00149815,
        0.00331502], dtype=float32), 'pqc_param_map': array([[ 0,  0,  0,  6],
       [ 0,  0,  1,  7],
       [ 0,  0,  2,  8],
       [ 0,  1,  0,  9],
       [ 0,  1,  1, 10],
       [ 0,  1,  2, 11],
       [ 1,  0,  0, 18],
  

## Test Template Update with Random PQC Parameters

In [10]:
# Generate random PQC parameters
pqc_params = np.random.randn(PQC_BLOCKS, NUM_QUBITS, 3).astype(np.float32)
print(f"PQC Parameters shape: {pqc_params.shape}")
print(f"Expected: ({PQC_BLOCKS}, {NUM_QUBITS}, 3)")

# Update template with new parameters
gate_ids, wire1, wire2, theta = update_pqc_circuit_template(pqc_noisy_circ_template, pqc_params)

print(f"\nUpdated Circuit:")
print(f"  Total gates: {len(gate_ids)}")
print(f"  Logical gates: {NUM_GATES}")
print(f"  Noise gates: {len(noisy_circ) - NUM_GATES}")
print(f"  PQC gates: {PQC_BLOCKS * NUM_QUBITS * 3}")
print(f"  Expected total: {len(noisy_circ) + PQC_BLOCKS * NUM_QUBITS * 3}")

PQC Parameters shape: (2, 2, 3)
Expected: (2, 2, 3)

Updated Circuit:
  Total gates: 26
  Logical gates: 4
  Noise gates: 10
  PQC gates: 12
  Expected total: 26


## Performance Comparison: Template vs Full Rebuild

In [11]:
import time
from pqcqec.noise.builder import build_circuit_with_pqc

num_iterations = 10000

# Method 1: Template updates
params_list = [np.random.randn(PQC_BLOCKS, NUM_QUBITS, 3).astype(np.float32) for _ in range(num_iterations)]

start = time.perf_counter()
for params in params_list:
    g, w1, w2, theta = update_pqc_circuit_template(pqc_noisy_circ_template, params)
end = time.perf_counter()
template_time = (end - start) / num_iterations

print(f"Template Update Method:")
print(f"  Average time: {template_time*1000:.4f} ms")
print(f"  Throughput: {1/template_time:.0f} updates/sec")

# Method 2: Full rebuild
start = time.perf_counter()
for params in params_list:
    g, w1, w2, theta = build_circuit_with_pqc(
        noisy_circ, NUM_QUBITS, NUM_GATE_BLOCKS, ['rz', 'rx', 'rz'], params,
        return_numba=True, ignore_noise_gates=True
    )
end = time.perf_counter()
rebuild_time = (end - start) / num_iterations

print(f"\nFull Rebuild Method:")
print(f"  Average time: {rebuild_time*1000:.4f} ms")
print(f"  Throughput: {1/rebuild_time:.0f} updates/sec")

speedup = rebuild_time / template_time
print(f"\n{'='*60}")
print(f"SPEEDUP: {speedup:.1f}x faster with template!")
print(f"{'='*60}")

Template Update Method:
  Average time: 0.0021 ms
  Throughput: 478088 updates/sec

Full Rebuild Method:
  Average time: 0.0239 ms
  Throughput: 41767 updates/sec

SPEEDUP: 11.4x faster with template!
